# Galaxy-X-os  —  One-Click Colab Pipeline (SCALE x ODYSSEY)

Classify raw astronomical images into **5 celestial categories** with **Ensemble: ConvNeXt-Base + Swin-B + EfficientNet-B3**.

**How to run:** `Runtime → Change runtime type → GPU (T4)`, then `Runtime → Run all`.

Pipeline: clone → install → prepare data → train (3 backbones) → evaluate ensemble → Grad-CAM → download results.

Every cell is idempotent — re-running is safe.

## Cell 1 — Clone repo + install dependencies

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Srujan0798/Galaxy-X-os"
REPO_DIR = "/content/Galaxy-X-os"

# Clone if missing; otherwise HARD-RESET to the latest main so re-runs ALWAYS
# pick up new commits. (A stale clone was silently running old code before.)
if os.path.exists(os.path.join(REPO_DIR, "src/prepare_data.py")):
    print("Repo present -> fetching latest main (hard reset)...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=False)
elif os.path.exists("src/prepare_data.py"):
    REPO_DIR = os.getcwd()
    print(f"Inside repo at {REPO_DIR} -> fetching latest main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=False)
else:
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())
print("Latest commit:", subprocess.run(
    ["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
    capture_output=True, text=True).stdout.strip())

# Core deps + extras needed for the real-first data pipeline.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "astroNN", "kagglehub", "kaggle", "h5py"], check=False)
print("Dependencies installed.")

## Cell 1b — Verify GPU is present (fail loudly if not)

Training EfficientNet-B3 on CPU is impractically slow. If this fails, set
`Runtime → Change runtime type → Hardware accelerator → GPU` and re-run.

In [ ]:
import torch

if not torch.cuda.is_available():
    print("WARNING: No GPU detected. Training will be very slow on CPU.")
    print("  For best results, set Runtime -> Change runtime type -> GPU (T4)")
    print("  Continuing on CPU (3 epochs demo mode)...")
    device = "cpu"
else:
    device = "cuda"
print(f"Using device: {device}")

## Cell 2 - (Optional) Kaggle token for REAL nebula / cluster / planetary

Spiral & elliptical always come from real **Galaxy10** (no key needed).

For real **nebula / star_cluster / planetary**, provide your Kaggle token in the next cell:

- **Easiest:** paste the `KGAT_...` string into the box in the next cell.
- **Or:** add it as a Colab Secret named `KAGGLE_API_TOKEN` (key icon in the sidebar) and leave the box empty.
- Get a new token at: https://www.kaggle.com/settings -> *Create New Token*.

**Skipping this** is fine - those three classes fall back to clearly-labelled procedural
images so the pipeline never breaks. The choice is recorded honestly in
`data/processed/DATA_MANIFEST.json`.

> Token stays in **your** Colab session only. Rotate it on Kaggle after submission.


In [ ]:
# =========================================================================
# OPTIONAL: Kaggle token -> enables REAL nebula / star_cluster / planetary
# =========================================================================
# Spiral + elliptical always come from real Galaxy10 (no key needed).
#
# Three ways to provide the new KGAT_ token (only ONE is required):
#
#   (A) Paste it in the box below.  Easiest.
#         -> Set KAGGLE_API_TOKEN = "KGAT_your_token_here"
#
#   (B) Colab Secret (left sidebar -> key icon):
#         -> Name: KAGGLE_API_TOKEN
#         -> Value: KGAT_your_token_here
#         -> Leave the box below empty.
#
#   (C) Legacy kaggle.json (upload via Files panel on the left).
#
# SECURITY: nothing you paste here leaves this Colab session. Do NOT commit.
# Rotate the token on Kaggle (Settings -> Create New Token) after submission.
# =========================================================================

import os
from pathlib import Path

# ---- (A) Paste your token between the quotes (or leave empty) --------------
KAGGLE_API_TOKEN = ""
# ----------------------------------------------------------------------------

# ---- (B) Auto-pull from a Colab Secret named KAGGLE_API_TOKEN --------------
if not KAGGLE_API_TOKEN:
    try:
        from google.colab import userdata
        KAGGLE_API_TOKEN = userdata.get("KAGGLE_API_TOKEN") or ""
    except Exception:
        pass
# ----------------------------------------------------------------------------

# ---- Install the token for the kagglehub / kaggle CLI ----------------------
if KAGGLE_API_TOKEN:
    token = KAGGLE_API_TOKEN.strip()
    os.environ["KAGGLE_API_TOKEN"] = token
    kdir = Path.home() / ".kaggle"
    kdir.mkdir(exist_ok=True)
    (kdir / "access_token").write_text(token + "\n")
    os.chmod(kdir / "access_token", 0o600)
    print("Kaggle token installed -> REAL nebula/cluster/planetary will be attempted.")
else:
    print("No Kaggle token -> procedural fallback for nebula/cluster/planetary.")
# ----------------------------------------------------------------------------

# ---- Status summary --------------------------------------------------------
has_creds = (
    bool(KAGGLE_API_TOKEN)
    or (Path.home() / ".kaggle" / "kaggle.json").exists()
)
print("Kaggle credentials present:", has_creds)
# ----------------------------------------------------------------------------


## Cell 3 — Prepare data (real-first, safe-fallback, idempotent)

Builds `data/processed/{train,val,test}/<class>/`, runs a disjoint 80/10/10
stratified split with an MD5 leakage check, and writes `DATA_MANIFEST.json`
+ `class_weights.json`.

In [ ]:
!python src/prepare_data.py --per-class 500

import json
with open("data/processed/DATA_MANIFEST.json") as f:
    print(json.dumps(json.load(f), indent=2))

## Cell 4 — Train (full EfficientNet-B3 fine-tune on the GPU)

Writes `checkpoints/best_model.pth` (best val accuracy). Target: **> 80%** val accuracy (problem-statement minimum). Reported final run: **93.17%** test / **92.77%** TTA on Colab T4.
Uses `configs/config.yaml` (progressive unfreezing, OneCycleLR, mixed precision, early stopping).

In [ ]:
!python src/train.py

import torch
ckpt = torch.load("checkpoints/best_model.pth", map_location="cpu", weights_only=True)
bva = ckpt.get("best_val_acc", None)
if bva is not None:
    print(f"\nBest val accuracy: {bva:.4f} (epoch {ckpt.get('epoch', '?') + 1})")
else:
    print("\nCheckpoint loaded but no best_val_acc found.")

## Cell 4b — Train ConvNeXt-Base (88M params)

Second backbone for the ensemble. Trains on the same data split.
Writes `checkpoints/convnext_base.pth`.

In [ ]:
!python src/train.py --backbone convnext_base --checkpoint checkpoints/best_model_convnext_base.pth --epochs 3 --lr 3e-4 --batch-size 32 --label-smoothing 0.1 --focal-gamma 2.0

import torch
ckpt = torch.load("checkpoints/best_model_convnext_base.pth", map_location="cpu", weights_only=True)
bva = ckpt.get("best_val_acc", None)
if bva is not None:
    print(f"ConvNeXt-Base: {ckpt.get('epoch', '?')} epochs, best_val_acc={bva:.4f}")
else:
    print("Checkpoint not found or incomplete — run train cell first.")

## Cell 4c — Train Swin-B (88M params)

Third backbone for the ensemble.
Writes `checkpoints/swin_base.pth`.

In [ ]:
!python src/train.py --backbone swin_base_patch4_window7_224 --checkpoint checkpoints/best_model_swin_base_patch4_window7_224.pth --epochs 3 --lr 3e-4 --batch-size 32 --label-smoothing 0.1 --focal-gamma 2.0

import torch
ckpt = torch.load("checkpoints/best_model_swin_base_patch4_window7_224.pth", map_location="cpu", weights_only=True)
bva = ckpt.get("best_val_acc", None)
if bva is not None:
    print(f"Swin-B: {ckpt.get('epoch', '?')} epochs, best_val_acc={bva:.4f}")
else:
    print("Checkpoint not found or incomplete — run train cell first.")

## Cell 4d — Train EfficientNet-B3 (11.6M params)

Third backbone (or standalone if not using ensemble).
Writes `checkpoints/efficientnet_b3.pth`.

In [ ]:
!python src/train.py --backbone efficientnet_b3 --checkpoint checkpoints/best_model_efficientnet_b3.pth --epochs 3 --lr 3e-4 --batch-size 32 --label-smoothing 0.1 --focal-gamma 2.0

import torch
ckpt = torch.load("checkpoints/best_model_efficientnet_b3.pth", map_location="cpu", weights_only=True)
bva = ckpt.get("best_val_acc", None)
if bva is not None:
    print(f"EfficientNet-B3: {ckpt.get('epoch', '?')} epochs, best_val_acc={bva:.4f}")
else:
    print("Checkpoint not found or incomplete — run train cell first.")

## Cell 5 — Evaluate (standard + TTA + Ensemble)

Evaluates single model, then ensemble with advanced TTA.
Shows `results/evaluation_results.json` with accuracy, F1, uncertainty metrics.

In [ ]:
# Single model evaluation
!python src/evaluate.py --tta standard

# Ensemble evaluation (if all 3 checkpoints exist)
import os
if all(os.path.exists(f"checkpoints/{m}.pth") for m in ["convnext_base", "swin_base", "efficientnet_b3"]):
    !python src/evaluate.py --ensemble --tta advanced --overwrite
    print("Ensemble evaluation complete!")
else:
    print("Ensemble checkpoints not all present, skipping ensemble eval.")

import json
from IPython.display import Image as IPyImage, display
with open("results/evaluation_results.json") as f:
    results = json.load(f)
print(f"\nStandard: Acc={results['standard']['accuracy']:.4f} F1={results['standard']['macro_f1']:.4f}")
if results.get('tta'):
    print(f"TTA:      Acc={results['tta']['accuracy']:.4f} F1={results['tta']['macro_f1']:.4f}")
if results.get('uncertainty'):
    print(f"Uncertainty: epistemic={results['uncertainty']['mean_epistemic']:.6f} aleatoric={results['uncertainty']['mean_aleatoric']:.6f}")
display(IPyImage(filename="results/confusion_matrix.png"))
display(IPyImage(filename="results/per_class_metrics.png"))

## Cell 5 — Evaluate (standard + Test-Time Augmentation)

Shows `results/evaluation_results.json` and the confusion matrix inline.

In [ ]:
!python src/evaluate.py

import json
from IPython.display import Image as IPyImage, display
with open("results/evaluation_results.json") as f:
    print(json.dumps(json.load(f), indent=2))
for p in ["results/confusion_matrix.png", "results/per_class_metrics.png"]:
    try:
        display(IPyImage(filename=p))
    except Exception as e:
        print(f"(could not display {p}: {e})")

## Cell 6 — Grad-CAM from the REAL trained model

In [ ]:
!python src/gradcam.py

import glob
from IPython.display import Image as IPyImage, display
cams = sorted(glob.glob("results/gradcam/*.png"))
print(f"Generated {len(cams)} Grad-CAM images.")
for p in cams[:4]:
    display(IPyImage(filename=p))

## Cell 7 — Zip results + checkpoint and download

After download: **unzip `results.zip` into the repo root, then commit.**

In [ ]:
import zipfile, os

with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk("results"):
        for fn in fnames:
            fp = os.path.join(root, fn)
            z.write(fp, fp)
    for ckpt in ["best_model.pth", "best_model_convnext_base.pth", "best_model_swin_base_patch4_window7_224.pth", "best_model_efficientnet_b3.pth"]:
        if os.path.exists(f"checkpoints/{ckpt}"):
            z.write(f"checkpoints/{ckpt}", f"checkpoints/{ckpt}")
    # Include the honesty record + class weights so they come back with results.
    for extra in ("data/processed/DATA_MANIFEST.json", "data/processed/class_weights.json"):
        if os.path.exists(extra):
            z.write(extra, extra)

print("Wrote results.zip:", round(os.path.getsize("results.zip") / 1e6, 1), "MB")
print("\nAfter download:\n  1. unzip results.zip into the repo root\n"
      "  2. git add results checkpoints data/processed/DATA_MANIFEST.json && git commit")

try:
    from google.colab import files
    files.download("results.zip")
except Exception as e:
    print(f"(auto-download unavailable: {e} — download results.zip from the Files panel)")